In [1]:
"""
Step 06: Daily Feature And Target (TSDAY) Data Collection (shared by exp02, exp03 and later daily experiments)

Objective:

    - One row per session, evaluated at the DECISION BAR (15:58; the second-to-last bar on half days). The trade it labels
      fills at the open of the next bar (15:59).
    - 16 scale-free daily features (so.config.DAILY_FEATURE_COL_STR_LIST): previous-day return and range, intraday return,
      20-session volatility, distance from the all-time high, past returns (5 / 20 / 60 / 120 / 250 sessions), distances
      from the 50- and 200-session moving averages, volatility ratio (20 / 60), distances from the 60- and 250-session
      highs, sessions since the 60-session high (definitions in so/features/daily_features.py).
    - Target: y_fwd_positive = 1 if buying at the 15:59 fill and selling at the close 20 sessions later is profitable after
      costs (fwd_net_return_20d > 0). The last 20 sessions have no label yet (NaN).

Built directly from the minute bars in a few seconds. Experiments rebuild the same table in memory (so they always use the
current raw data); this file is the saved record. No result is computed here, and nothing is logged.
Prerequisite: the raw data only (steps 01-05 are not needed). Output: store02_workzone/step06_TSDAY_data/TSDAY_data.csv.
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
# IMPORT THE SHARED CONFIGURATION AND THE PATHS
from so import config, paths
# IMPORT DATA LOADING, EXECUTION AND DAILY FEATURE FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
from so.features.daily_features import get_daily_feature_pdf
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path

In [3]:
# CALL FUNCTION TO READ THE COMPLETE MINUTE BARS (RAW FOLDER OF so.paths)
complete_ohlcv_pdf = get_complete_ohlcv_pdf()
# CALL FUNCTION TO CONVERT THE COMPLETE DATA TO ARRAYS
ohlcv_array_dict = get_ohlcv_array_dict(complete_ohlcv_pdf)
# CALL FUNCTION TO BUILD THE DAILY FEATURES AND TARGET
daily_pdf = get_daily_feature_pdf(ohlcv_array_dict)
# DISPLAY INFORMATION
print(f"min timestamp:\t{complete_ohlcv_pdf.timestamp.min()}")
print(f"max timestamp:\t{complete_ohlcv_pdf.timestamp.max()}")
print(f"Sessions:\t{len(daily_pdf):,}\tLabelled:\t{int(daily_pdf['y_fwd_positive'].notna().sum()):,}")

min timestamp:	2005-01-03 09:30:00-05:00
max timestamp:	2026-08-13 15:59:00-04:00
Sessions:	5,436	Labelled:	5,416


In [4]:
"""
Missing values: the long lookbacks (250-session return and high, 200-session moving average) are empty for the first year
of data, and the label is empty for the last 20 sessions. Rows with any missing feature or label are never used to train.
(No result table is shown here on purpose: base rates and returns by period belong to the experiments, which control which
period they look at.)
"""
# DISPLAY THE MISSING VALUES PER FEATURE AND THE FIRST COMPLETE SESSION
display(daily_pdf[config.DAILY_FEATURE_COL_STR_LIST + ["y_fwd_positive"]].isna().sum().rename("missing_count").to_frame().T)
first_complete_date = daily_pdf.dropna(subset=config.DAILY_FEATURE_COL_STR_LIST)["date"].iloc[0]
print(f"First session with every feature:\t{first_complete_date}")
# PREVIEW THE FIRST COMPLETE ROWS
daily_pdf[daily_pdf["date"] >= first_complete_date].head()

,prev_day_return_pct,prev_day_range_pct,intraday_return_pct,daily_volatility,ath_drawdown_pct,return_5d,return_20d,return_60d,return_120d,return_250d,ma50_dist_pct,ma200_dist_pct,volatility_ratio_20_60,high250_dist_pct,high60_dist_pct,sessions_since_high60,y_fwd_positive
missing_count,2,1,0,21,0,5,20,60,120,250,50,200,61,249,59,59,20


First session with every feature:	2005-12-29


,session_idx,date,decision_ts,decision_idx,fill_idx,decision_close,fill_open,high_to_decision,session_open,session_high,...,return_60d,return_120d,return_250d,ma50_dist_pct,ma200_dist_pct,high60_dist_pct,high250_dist_pct,sessions_since_high60,fwd_net_return_20d,y_fwd_positive
250,250,2005-12-29,2005-12-29 15:58:00-05:00,97704,97705,125.30,125.31,125.96,125.72,125.96,...,0.032295,0.026881,0.043645,0.010813,0.034461,-0.033850,-0.033850,22.0,0.025455,1.0
251,251,2005-12-30,2005-12-30 15:58:00-05:00,98094,98095,124.65,124.64,125.06,124.80,125.06,...,0.041876,0.019882,0.049773,0.004369,0.028834,-0.038862,-0.038862,23.0,0.026314,1.0
252,252,2006-01-03,2006-01-03 15:58:00-05:00,98484,98485,126.72,126.73,127.00,125.10,127.00,...,0.063533,0.035802,0.070994,0.020241,0.045698,-0.022901,-0.022901,24.0,0.011599,1.0
253,253,2006-01-04,2006-01-04 15:58:00-05:00,98874,98875,127.22,127.22,127.49,126.86,127.49,...,0.063090,0.036500,0.071687,0.022779,0.049480,-0.019045,-0.019045,25.0,-0.001100,0.0
254,254,2006-01-05,2006-01-05 15:58:00-05:00,99264,99265,127.20,127.19,127.59,127.17,127.59,...,0.070978,0.036084,0.072874,0.021100,0.048925,-0.019200,-0.019200,26.0,-0.006368,0.0


In [5]:
# SAVE THE DAILY FEATURE AND TARGET TABLE (OVERWRITE: THE TABLE IS REBUILT FROM THE RAW DATA EVERY TIME)
write_csv_file_to_path(daily_pdf, f"{paths.LOCAL_TSDAY_DATA_FILE_PATH_STR}TSDAY_data.csv", "W")

ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store02_workzone/step06_TSDAY_data/TSDAY_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store02_workzone/step06_TSDAY_data/TSDAY_data.csv' Has Been Saved!
